# Base 1 — Bitcoin diário

Pipeline SARIMAX adaptada de `docs/Pipes de exemplo/atividade_sarimax.ipynb`. Resultados ficam incorporados neste notebook; nenhum CSV/JSON auxiliar é criado.

In [1]:
from pathlib import Path
import itertools, platform, time, warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
BASE_NUMBER=1
TIME_COL='Date'
TARGET_COL='Close'
FREQUENCY='D'
TRAIN_RATIO=0.7
SEASONAL_PERIOD=7
ACF_LAGS=30
LJUNG_LAGS=[1, 7, 14, 30]
RANDOM_STATE=42
SMOKE_TEST=True
np.random.seed(RANDOM_STATE)

In [3]:
def encontrar_grupo(numero):
    atual = Path.cwd().resolve()
    for raiz in [atual, *atual.parents]:
        pasta = raiz / "trabalho" / "bases" / f"grupo{numero}"
        if pasta.is_dir(): return pasta
        pasta = raiz / f"grupo{numero}"
        if pasta.is_dir(): return pasta
    raise FileNotFoundError(f"grupo{numero} não encontrado")

def carregar_base(pasta):
    treino = pasta / f"base{BASE_NUMBER}_treino_preparada.csv"
    teste = pasta / f"base{BASE_NUMBER}_teste_preparada.csv"
    if treino.exists() and teste.exists():
        return pd.concat([pd.read_csv(treino), pd.read_csv(teste)], ignore_index=True)
    for nome in [f"grupo{BASE_NUMBER}.csv", "grupo5_new.csv"]:
        arquivo = pasta / nome
        if arquivo.exists(): return pd.read_csv(arquivo)
    raise FileNotFoundError("CSV da base não encontrado")

def mape_seguro(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float); m = y != 0
    return float(np.mean(np.abs((y[m]-p[m])/y[m]))*100) if m.any() else np.nan

def metricas(y, p):
    y, p = np.asarray(y,float), np.asarray(p,float)
    return {"MAE": mean_absolute_error(y,p), "RMSE": mean_squared_error(y,p)**.5,
            "MedAE": median_absolute_error(y,p), "R2": r2_score(y,p), "MAPE_pct": mape_seguro(y,p)}

def escalar_exog(x_treino, x_futuro):
    if x_treino is None: return None, None, None
    scaler = StandardScaler()
    a = pd.DataFrame(scaler.fit_transform(x_treino), index=x_treino.index, columns=x_treino.columns)
    b = pd.DataFrame(scaler.transform(x_futuro), index=x_futuro.index, columns=x_futuro.columns)
    return a, b, scaler

def ajustar(y, x, order, seasonal_order, trend="n", maxiter=100):
    inicio=time.perf_counter()
    with warnings.catch_warnings(record=True) as capturados:
        warnings.simplefilter("always")
        resultado=SARIMAX(y, exog=x, order=order, seasonal_order=seasonal_order, trend=trend,
                          enforce_stationarity=False, enforce_invertibility=False).fit(disp=False,maxiter=maxiter)
    return resultado, bool(resultado.mle_retvals.get("converged",True)), time.perf_counter()-inicio, " | ".join(str(w.message) for w in capturados[:3])

def ajustar_prever(treino, futuro, order, seasonal_order, exog_cols=(), trend="n"):
    xt, xf, scaler = escalar_exog(treino[list(exog_cols)] if exog_cols else None,
                                  futuro[list(exog_cols)] if exog_cols else None)
    res, convergiu, segundos, avisos = ajustar(treino[TARGET_COL], xt, order, seasonal_order, trend)
    pred = res.get_forecast(steps=len(futuro), exog=xf).predicted_mean
    pred.index=futuro.index
    return {"resultado":res,"previsao":pred,"convergiu":convergiu,"segundos":segundos,"avisos":avisos,"scaler":scaler}

## 1. SARIMAX — ingestão, corte temporal e preparação das exógenas

In [4]:
BASE_DIR=encontrar_grupo(BASE_NUMBER)
df=carregar_base(BASE_DIR)
df[TIME_COL]=pd.to_datetime(df[TIME_COL],errors="coerce",dayfirst=(BASE_NUMBER==4))
df[TARGET_COL]=pd.to_numeric(df[TARGET_COL],errors="coerce")
df=df.dropna(subset=[TIME_COL]).sort_values(TIME_COL)
numericas=df.select_dtypes(include=np.number).columns.tolist()
df=df.groupby(TIME_COL,sort=True)[numericas].mean()
grade=pd.date_range(df.index.min(),df.index.max(),freq=FREQUENCY)
df=df.reindex(grade)
df["hour_sin"]=np.sin(2*np.pi*df.index.hour/24); df["hour_cos"]=np.cos(2*np.pi*df.index.hour/24)
df["dow_sin"]=np.sin(2*np.pi*df.index.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*df.index.dayofweek/7)
df["month_sin"]=np.sin(2*np.pi*(df.index.month-1)/12); df["month_cos"]=np.cos(2*np.pi*(df.index.month-1)/12)
EXOG_COLS=["hour_sin","hour_cos","dow_sin","dow_cos","month_sin","month_cos"]
corte=int(df[TARGET_COL].notna().sum()*TRAIN_RATIO)
inicio_teste=df[TARGET_COL].dropna().index[corte]
treino=df.loc[df.index<inicio_teste].copy(); teste=df.loc[df.index>=inicio_teste].copy()
teste_observado=teste.dropna(subset=[TARGET_COL]).copy()
print("Base:",BASE_DIR); print("Treino:",treino.index.min(),treino.index.max(),treino[TARGET_COL].notna().sum())
print("Teste:",teste_observado.index.min(),teste_observado.index.max(),len(teste_observado))
display(df[[TARGET_COL,*EXOG_COLS]].head())

Base: C:\Users\matheuscastilho-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech 3 Ano\Inovação e Technologia\Projetos de exemplo da aula\Projeto aula\trabalho-daniel-modelos\trabalho\bases\grupo1
Treino: 2017-01-01 00:00:00 2022-06-08 00:00:00 1985
Teste: 2022-06-09 00:00:00 2024-10-06 00:00:00 851


,Close,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos
2017-01-01,998.325012,0.0,1.0,-0.781831,0.623490,0.0,1.0
2017-01-02,1021.750000,0.0,1.0,0.000000,1.000000,0.0,1.0
2017-01-03,1043.839966,0.0,1.0,0.781831,0.623490,0.0,1.0
2017-01-04,1154.729980,0.0,1.0,0.974928,-0.222521,0.0,1.0
2017-01-05,1013.380005,0.0,1.0,0.433884,-0.900969,0.0,1.0


In [5]:
serie=treino[TARGET_COL].dropna()
print("ADF p-valor:",adfuller(serie,autolag="AIC")[1]); print("KPSS p-valor:",kpss(serie,regression="c",nlags="auto")[1])
lags=min(ACF_LAGS,max(1,len(serie)//3))
fig,ax=plt.subplots(1,2,figsize=(14,4)); plot_acf(serie,lags=lags,ax=ax[0]); plot_pacf(serie,lags=min(lags,50),method="ywm",ax=ax[1]); plt.show()

ADF p-valor: 0.5985903949446029
KPSS p-valor: 0.01


C:\Users\matheuscastilho-ieg\AppData\Local\Temp\ipykernel_28612\4203662489.py:2: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  print("ADF p-valor:",adfuller(serie,autolag="AIC")[1]); print("KPSS p-valor:",kpss(serie,regression="c",nlags="auto")[1])
C:\Users\matheuscastilho-ieg\AppData\Local\Temp\ipykernel_28612\4203662489.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(1,2,figsize=(14,4)); plot_acf(serie,lags=lags,ax=ax[0]); plot_pacf(serie,lags=min(lags,50),method="ywm",ax=ax[1]); plt.show()


## 2. Otimização SARIMAX — ordens, sazonalidade, BIC e convergência

In [6]:
validacao_n=min(max(2*SEASONAL_PERIOD,20),max(20,len(treino)//5))
ajuste=treino.iloc[:-validacao_n]; validacao=treino.iloc[-validacao_n:]
if SMOKE_TEST:
    ordens=[(1,0,1),(1,1,1),(2,1,1)]
    sazonais=[(0,0,0,0),(1,0,1,SEASONAL_PERIOD)]
else:
    ordens=list(itertools.product(range(3),range(2),range(3)))
    sazonais=[(0,0,0,0)]+[(P,D,Q,SEASONAL_PERIOD) for P,D,Q in itertools.product(range(2),range(2),range(2))]
conjuntos={"SARIMA_sem_exogenas":[],"SARIMAX_calendario":EXOG_COLS}
linhas=[]
for order,seasonal_order,(familia,cols) in itertools.product(ordens,sazonais,conjuntos.items()):
    try:
        out=ajustar_prever(ajuste,validacao,order,seasonal_order,cols)
        alvo=validacao[TARGET_COL].notna(); score=metricas(validacao.loc[alvo,TARGET_COL],out["previsao"].loc[alvo])
        linhas.append({"familia":familia,"order":order,"seasonal_order":seasonal_order,"exogenas":cols,
                       "BIC":out["resultado"].bic,"convergiu":out["convergiu"],"segundos":out["segundos"],"avisos":out["avisos"],**score})
    except Exception as erro:
        linhas.append({"familia":familia,"order":order,"seasonal_order":seasonal_order,"exogenas":cols,"erro":str(erro),"MAE":np.inf})
busca=pd.DataFrame(linhas).sort_values(["MAE","BIC"],na_position="last").reset_index(drop=True)
display(busca)
validos=busca[(busca["MAE"]<np.inf)&busca["convergiu"].fillna(False)]
melhor=(validos if len(validos) else busca[busca["MAE"]<np.inf]).iloc[0]
display(melhor.to_frame("melhor configuração"))

,familia,order,seasonal_order,exogenas,BIC,convergiu,segundos,avisos,MAE,RMSE,MedAE,R2,MAPE_pct
0,SARIMA_sem_exogenas,"(2, 1, 1)","(0, 0, 0, 0)",[],32530.739286,True,1.485295,,807.597464,936.475769,796.397662,-0.086892,2.696047
1,SARIMA_sem_exogenas,"(1, 0, 1)","(0, 0, 0, 0)",[],32539.750696,True,0.505652,,807.753410,948.215272,770.321414,-0.114313,2.691148
2,SARIMA_sem_exogenas,"(1, 0, 1)","(1, 0, 1, 7)",[],32441.093285,True,2.074372,,809.893009,945.231724,746.702423,-0.107312,2.698303
3,SARIMAX_calendario,"(2, 1, 1)","(0, 0, 0, 0)","[hour_sin, hour_cos, dow_sin, dow_cos, month_s...",32579.567511,True,6.749743,,814.081044,948.629486,837.336518,-0.115287,2.718596
4,SARIMAX_calendario,"(1, 1, 1)","(0, 0, 0, 0)","[hour_sin, hour_cos, dow_sin, dow_cos, month_s...",32568.579434,True,0.456265,,814.261015,948.767397,837.872368,-0.115611,2.719248
5,SARIMAX_calendario,"(1, 1, 1)","(1, 0, 1, 7)","[hour_sin, hour_cos, dow_sin, dow_cos, month_s...",32469.614644,True,3.580071,,815.821062,946.937423,803.889287,-0.111312,2.723105
6,SARIMA_sem_exogenas,"(1, 1, 1)","(1, 0, 1, 7)",[],32425.567133,True,1.045496,,820.234562,943.572458,794.936010,-0.103428,2.739380
7,SARIMA_sem_exogenas,"(1, 1, 1)","(0, 0, 0, 0)",[],32524.321711,True,0.069738,,820.879151,946.400038,829.263447,-0.110051,2.742935
8,SARIMAX_calendario,"(2, 1, 1)","(1, 0, 1, 7)","[hour_sin, hour_cos, dow_sin, dow_cos, month_s...",32473.579218,True,11.764215,,872.985143,1061.419821,717.656458,-0.396265,2.875826
9,SARIMA_sem_exogenas,"(2, 1, 1)","(1, 0, 1, 7)",[],32434.747680,True,2.726756,,940.900236,1153.708246,758.668641,-0.649625,3.094461


,melhor configuração
familia,SARIMA_sem_exogenas
order,"(2, 1, 1)"
seasonal_order,"(0, 0, 0, 0)"
exogenas,[]
BIC,32530.739286
convergiu,True
segundos,1.485295
avisos,
MAE,807.597464
RMSE,936.475769


## 3. MAE e demais métricas SARIMAX

In [7]:
final=ajustar_prever(treino,teste,tuple(melhor.order),tuple(melhor.seasonal_order),melhor.exogenas)
pred=final["previsao"].loc[teste_observado.index]
previsoes=pd.DataFrame({"y_true":teste_observado[TARGET_COL],"y_pred_sarimax":pred})
previsoes["residuo_sarimax"]=previsoes.y_true-previsoes.y_pred_sarimax
previsoes["erro_absoluto_sarimax"]=previsoes.residuo_sarimax.abs()
resumo={**metricas(previsoes.y_true,previsoes.y_pred_sarimax),"BIC":final["resultado"].bic,
        "convergiu":final["convergiu"],"segundos":final["segundos"],"order":melhor.order,
        "seasonal_order":melhor.seasonal_order,"familia":melhor.familia,"exogenas":melhor.exogenas}
display(pd.Series(resumo,name="valor").to_frame())

,valor
MAE,14366.495393
RMSE,18992.789672
MedAE,10201.437032
R2,-0.144354
MAPE_pct,36.512907
BIC,32866.187785
convergiu,True
segundos,2.857943
order,"(2, 1, 1)"
seasonal_order,"(0, 0, 0, 0)"


## 4. Previsão e resíduos individuais SARIMAX

In [8]:
fig,ax=plt.subplots(figsize=(13,5)); ax.plot(previsoes.index,previsoes.y_true,label="Real",color="black"); ax.plot(previsoes.index,previsoes.y_pred_sarimax,label="SARIMAX",color="#e25c3e"); ax.legend(); ax.set_title(f"Base {BASE_NUMBER} — previsão versus real"); plt.show()
display(previsoes.head(20))

C:\Users\matheuscastilho-ieg\AppData\Local\Temp\ipykernel_28612\3773667800.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig,ax=plt.subplots(figsize=(13,5)); ax.plot(previsoes.index,previsoes.y_true,label="Real",color="black"); ax.plot(previsoes.index,previsoes.y_pred_sarimax,label="SARIMAX",color="#e25c3e"); ax.legend(); ax.set_title(f"Base {BASE_NUMBER} — previsão versus real"); plt.show()


,y_true,y_pred_sarimax,residuo_sarimax,erro_absoluto_sarimax
2022-06-09,30111.998047,30243.920993,-131.922946,131.922946
2022-06-10,29083.804688,30243.215280,-1159.410592,1159.410592
2022-06-11,28360.810547,30243.009778,-1882.199231,1882.199231
2022-06-12,26762.648438,30243.241594,-3480.593157,3480.593157
2022-06-13,22487.388672,30243.011453,-7755.622781,7755.622781
2022-06-14,22206.792969,30243.239061,-8036.446092,8036.446092
2022-06-15,22572.839844,30243.013986,-7670.174143,7670.174143
2022-06-16,20381.650391,30243.236555,-9861.586164,9861.586164
2022-06-17,20471.482422,30243.016464,-9771.534043,9771.534043
2022-06-18,19017.642578,30243.234104,-11225.591526,11225.591526


## 5. ACF e diagnóstico dos resíduos SARIMAX

In [9]:
residuos=previsoes.residuo_sarimax.dropna(); fig,ax=plt.subplots(2,2,figsize=(13,8)); ax[0,0].plot(residuos); ax[0,0].axhline(0,color="black",ls="--"); plot_acf(residuos,lags=min(ACF_LAGS,len(residuos)//3),ax=ax[0,1]); stats.probplot(residuos,dist="norm",plot=ax[1,0]); ax[1,1].hist(residuos,bins=30,density=True); plt.tight_layout(); plt.show()

C:\Users\matheuscastilho-ieg\AppData\Local\Temp\ipykernel_28612\1993936898.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  residuos=previsoes.residuo_sarimax.dropna(); fig,ax=plt.subplots(2,2,figsize=(13,8)); ax[0,0].plot(residuos); ax[0,0].axhline(0,color="black",ls="--"); plot_acf(residuos,lags=min(ACF_LAGS,len(residuos)//3),ax=ax[0,1]); stats.probplot(residuos,dist="norm",plot=ax[1,0]); ax[1,1].hist(residuos,bins=30,density=True); plt.tight_layout(); plt.show()


## 6. Ljung–Box SARIMAX

In [10]:
lags=[x for x in LJUNG_LAGS if x<len(residuos)]; ljung=acorr_ljungbox(residuos,lags=lags,return_df=True); ljung["rejeita_ruido_branco_5pct"]=ljung.lb_pvalue<.05; display(ljung)

,lb_stat,lb_pvalue,rejeita_ruido_branco_5pct
1,848.301522,1.701451e-186,True
7,5844.773782,0.000000e+00,True
14,11451.275963,0.000000e+00,True
30,23442.019466,0.000000e+00,True


## 7. Coeficientes das variáveis exógenas

In [11]:
parametros=final["resultado"].params; coef=parametros[parametros.index.isin(EXOG_COLS)].to_frame("coeficiente"); display(coef if len(coef) else pd.DataFrame({"observação":["Modelo selecionado sem exógenas."]}))

,observação
0,Modelo selecionado sem exógenas.
